In [0]:
from pyspark.sql.functions import col, to_timestamp

In [0]:
catalog = "medallion_trial"

In [0]:
print("Starting Silver Layer Transformations...")

# ==========================================
# 1. Transform the Orders Table
# ==========================================
df_raw_orders = spark.table(f"{catalog}.bronze.raw_orders")

df_silver_orders = df_raw_orders \
    .withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp"))) \
    .withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date"))) \
    .dropDuplicates(["order_id"]) \
    .dropna(subset=["order_id", "customer_id"]) # Drop rows missing critical IDs

df_silver_orders.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.cleaned_orders")
    
print("✅ Cleaned Orders saved to Silver!")

In [0]:
# ==========================================
# 2. Transform the Order Items Table
# ==========================================
df_raw_items = spark.table(f"{catalog}.bronze.raw_order_items")

df_silver_items = df_raw_items \
    .withColumn("price", col("price").cast("float")) \
    .withColumn("freight_value", col("freight_value").cast("float")) \
    .dropDuplicates(["order_id", "order_item_id"])

df_silver_items.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.cleaned_order_items")

print("✅ Cleaned Order Items saved to Silver!")